## Introduction

This notebook provides a fully self-contained TensorFlow 2.x implementation of a Wasserstein GAN with Gradient Penalty (WGAN-GP) trained on MNIST.
All training and analysis utilities that previously lived in separate Python files are embedded here so you can run experiments and report results from a single place.
Feel free to tweak hyperparameters such as the learning rate, critic iterations, gradient-penalty strength, and model depth to study their impact on sample quality and training stability.

If you prefer running experiments outside the notebook, the same training loop is also available as a CLI script:
`python train_wgan_gp_tf.py --help` will show the available options for running locally, but for notebook work no separate script invocation is required.

In [ ]:
# === Environment & Utilities ===
import json
import math
from pathlib import Path
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from IPython.display import Image, display

AUTOTUNE = tf.data.AUTOTUNE


In [ ]:
# === Dataset preparation ===
def make_mnist_dataset(batch_size: int, buffer_size: int = 60000):
    """Load MNIST, scale to [-1, 1], and create a shuffled tf.data pipeline."""
    (x_train, _), _ = tf.keras.datasets.mnist.load_data()
    x_train = (x_train.astype("float32") / 127.5) - 1.0
    x_train = np.expand_dims(x_train, axis=-1)
    ds = tf.data.Dataset.from_tensor_slices(x_train)
    ds = ds.shuffle(buffer_size).batch(batch_size, drop_remainder=True)
    ds = ds.prefetch(AUTOTUNE)
    return ds


def get_mnist_dataset(batch_size: int = 64, shuffle_buffer: int = 60000):
    """Convenience wrapper used throughout the notebook."""
    return make_mnist_dataset(batch_size=batch_size, buffer_size=shuffle_buffer)



## Model Definitions
We build lightweight convolutional generator and critic networks with TensorFlow/Keras layers. Adjust the base channel count to control the model capacity.


In [ ]:
# === Models ===
def build_generator(latent_dim: int, base_channels: int = 64):
    model = tf.keras.Sequential(
        [
            layers.Input(shape=(latent_dim,)),
            layers.Dense(7 * 7 * base_channels, use_bias=False),
            layers.BatchNormalization(),
            layers.ReLU(),
            layers.Reshape((7, 7, base_channels)),
            layers.Conv2DTranspose(base_channels // 2, 4, strides=2, padding="same", use_bias=False),
            layers.BatchNormalization(),
            layers.ReLU(),
            layers.Conv2DTranspose(base_channels // 4, 4, strides=2, padding="same", use_bias=False),
            layers.BatchNormalization(),
            layers.ReLU(),
            layers.Conv2D(1, 5, padding="same", activation="tanh"),
        ],
        name="generator",
    )
    return model



def build_discriminator(base_channels: int = 64):
    model = tf.keras.Sequential(
        [
            layers.Input(shape=(28, 28, 1)),
            layers.Conv2D(base_channels, 5, strides=2, padding="same"),
            layers.LeakyReLU(0.2),
            layers.Conv2D(base_channels * 2, 5, strides=2, padding="same"),
            layers.LayerNormalization(),
            layers.LeakyReLU(0.2),
            layers.Flatten(),
            layers.Dense(1),
        ],
        name="discriminator",
    )
    return model



def create_models(latent_dim: int = 128, base_channels: int = 96):
    generator = build_generator(latent_dim=latent_dim, base_channels=base_channels)
    discriminator = build_discriminator(base_channels=base_channels)
    return generator, discriminator


In [ ]:
# === Training Utils ===
def preview_batch(dataset, n: int = 16):
    batch = next(iter(dataset.unbatch().batch(n)))
    images = (batch.numpy() + 1.0) * 0.5
    rows = int(math.sqrt(n))
    cols = rows
    fig, axes = plt.subplots(rows, cols, figsize=(cols, rows))
    for ax, img in zip(axes.flatten(), images):
        ax.imshow(img.squeeze(), cmap='gray')
        ax.axis('off')
    plt.show()



def save_image_grid(generator, latent_dim: int, grid_size: int, out_path: Path):
    noise = tf.random.normal([grid_size ** 2, latent_dim])
    preds = generator(noise, training=False)
    preds = (preds + 1.0) * 0.5
    preds = tf.clip_by_value(preds, 0.0, 1.0)
    canvas = np.zeros((28 * grid_size, 28 * grid_size))
    imgs = preds.numpy().reshape(grid_size ** 2, 28, 28)
    for i in range(grid_size):
        for j in range(grid_size):
            canvas[i * 28:(i + 1) * 28, j * 28:(j + 1) * 28] = imgs[i * grid_size + j]
    plt.figure(figsize=(grid_size, grid_size))
    plt.imshow(canvas, cmap='gray')
    plt.axis('off')
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path, bbox_inches='tight', pad_inches=0)
    plt.close()


In [ ]:
# === Core Trainers ===
class WGANGPTrainer:
    def __init__(
        self,
        latent_dim: int,
        batch_size: int,
        n_critic: int,
        gp_weight: float,
        gen: tf.keras.Model,
        disc: tf.keras.Model,
        g_opt: tf.keras.optimizers.Optimizer,
        d_opt: tf.keras.optimizers.Optimizer,
    ):
        self.latent_dim = latent_dim
        self.batch_size = batch_size
        self.n_critic = n_critic
        self.gp_weight = gp_weight
        self.generator = gen
        self.discriminator = disc
        self.g_opt = g_opt
        self.d_opt = d_opt

    def gradient_penalty(self, real_images, fake_images):
        epsilon = tf.random.uniform([self.batch_size, 1, 1, 1], 0.0, 1.0)
        interpolated = epsilon * real_images + (1.0 - epsilon) * fake_images
        with tf.GradientTape() as tape:
            tape.watch(interpolated)
            pred = self.discriminator(interpolated, training=True)
        grads = tape.gradient(pred, interpolated)
        grads = tf.reshape(grads, [self.batch_size, -1])
        slopes = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=1) + 1e-12)
        return tf.reduce_mean((slopes - 1.0) ** 2)

    @tf.function
    def critic_train_step(self, real_images):
        noise = tf.random.normal([self.batch_size, self.latent_dim])
        with tf.GradientTape() as tape:
            fake_images = self.generator(noise, training=True)
            fake_images = tf.stop_gradient(fake_images)
            real_logits = self.discriminator(real_images, training=True)
            fake_logits = self.discriminator(fake_images, training=True)
            gp = tf.constant(0.0)
            if self.gp_weight > 0:
                gp = self.gradient_penalty(real_images, fake_images)
            d_loss = tf.reduce_mean(fake_logits) - tf.reduce_mean(real_logits) + self.gp_weight * gp
        d_grads = tape.gradient(d_loss, self.discriminator.trainable_variables)
        self.d_opt.apply_gradients(zip(d_grads, self.discriminator.trainable_variables))
        wasserstein = tf.reduce_mean(real_logits) - tf.reduce_mean(fake_logits)
        return d_loss, wasserstein

    @tf.function
    def generator_train_step(self):
        noise = tf.random.normal([self.batch_size, self.latent_dim])
        with tf.GradientTape() as tape:
            fake_images = self.generator(noise, training=True)
            fake_logits = self.discriminator(fake_images, training=True)
            g_loss = -tf.reduce_mean(fake_logits)
        g_grads = tape.gradient(g_loss, self.generator.trainable_variables)
        self.g_opt.apply_gradients(zip(g_grads, self.generator.trainable_variables))
        return g_loss

    def sample_images(self, num_samples: int, out_path: Path):
        noise = tf.random.normal([num_samples, self.latent_dim])
        generated = self.generator(noise, training=False)
        generated = (generated + 1.0) * 0.5
        generated = tf.clip_by_value(generated, 0.0, 1.0)
        n = int(math.sqrt(num_samples))
        canvas = np.zeros((28 * n, 28 * n))
        imgs = generated.numpy().reshape(num_samples, 28, 28)
        for i in range(n):
            for j in range(n):
                canvas[i * 28 : (i + 1) * 28, j * 28 : (j + 1) * 28] = imgs[i * n + j]
        plt.figure(figsize=(n, n))
        plt.axis("off")
        plt.imshow(canvas, cmap="gray")
        out_path = Path(out_path)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(out_path, bbox_inches="tight", pad_inches=0)
        plt.close()


class NotebookTrainer:
    def __init__(
        self,
        generator: tf.keras.Model,
        discriminator: tf.keras.Model,
        dataset: tf.data.Dataset,
        batch_size: int,
        latent_dim: int = 128,
        n_critic: int = 3,
        gp_weight: float = 10.0,
        lr: float = 2e-4,
        beta1: float = 0.0,
        beta2: float = 0.99,
        output_dir: str = "outputs/notebook_run",
        sample_every: int = 5,
        sample_grid: int = 6,
        max_batches: int = 0,
        eager: bool = False,
    ):
        if eager:
            tf.config.run_functions_eagerly(True)
        self.dataset = dataset
        self.batch_size = batch_size
        self.latent_dim = latent_dim
        self.sample_every = sample_every
        self.sample_grid = sample_grid
        self.max_batches = max_batches
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

        g_opt = tf.keras.optimizers.Adam(lr, beta_1=beta1, beta_2=beta2)
        d_opt = tf.keras.optimizers.Adam(lr, beta_1=beta1, beta_2=beta2)

        self.trainer = WGANGPTrainer(
            latent_dim,
            batch_size,
            n_critic,
            gp_weight,
            generator,
            discriminator,
            g_opt,
            d_opt,
        )
        self.history: Dict[str, List[float]] = {"g_loss": [], "d_loss": [], "wasserstein": []}
        self.config = dict(
            latent_dim=latent_dim,
            n_critic=n_critic,
            gp_weight=gp_weight,
            lr=lr,
            beta1=beta1,
            beta2=beta2,
            batch_size=batch_size,
            sample_every=sample_every,
            sample_grid=sample_grid,
            max_batches=max_batches,
            output_dir=output_dir,
        )

    def train(self, epochs: int):
        for epoch in range(1, epochs + 1):
            g_epoch, d_epoch, w_epoch = [], [], []
            for step, real_images in enumerate(self.dataset):
                for _ in range(self.trainer.n_critic):
                    d_loss, w_dist = self.trainer.critic_train_step(real_images)
                    d_epoch.append(float(d_loss.numpy()))
                    w_epoch.append(float(w_dist.numpy()))
                g_loss = self.trainer.generator_train_step()
                g_epoch.append(float(g_loss.numpy()))
                if self.max_batches and (step + 1) >= self.max_batches:
                    break
            self.history['g_loss'].append(float(np.mean(g_epoch)))
            self.history['d_loss'].append(float(np.mean(d_epoch)))
            self.history['wasserstein'].append(float(np.mean(w_epoch)))
            if self.sample_every and (epoch % self.sample_every == 0 or epoch == epochs):
                save_image_grid(
                    self.trainer.generator,
                    self.latent_dim,
                    self.sample_grid,
                    Path(self.output_dir) / f"samples_epoch_{epoch:03d}.png",
                )
            print(
                f"Epoch {epoch}/{epochs} | G loss: {self.history['g_loss'][-1]:.4f} "
                f"| D loss: {self.history['d_loss'][-1]:.4f} | W-dist: {self.history['wasserstein'][-1]:.4f}"
            )
        with open(self.output_dir / "history.json", "w") as f:
            json.dump(self.history, f, indent=2)
        with open(self.output_dir / "config.json", "w") as f:
            json.dump(self.config | {"epochs": epochs}, f, indent=2)
        save_image_grid(self.trainer.generator, self.latent_dim, self.sample_grid, self.output_dir / "final_samples.png")
        return self.history


In [ ]:
# === History & Reporting Helpers ===
def load_history(path: Path):
    with open(path) as f:
        return json.load(f)


def plot_history(history, title: str = "WGAN-GP Training Curves", out_path: Path | None = None):
    epochs = range(1, len(history["g_loss"]) + 1)
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["g_loss"], label="Generator loss")
    plt.plot(epochs, history["d_loss"], label="Critic loss")
    plt.plot(epochs, history["wasserstein"], label="Wasserstein estimate")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    if out_path is not None:
        out_path = Path(out_path)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(out_path, bbox_inches="tight")
    plt.show()


In [ ]:
# === Notebook Experiment Runner ===
def run_notebook_experiment(config: Dict, title: str):
    print('Configuration:')
    print(json.dumps(config, indent=2))
    dataset = get_mnist_dataset(batch_size=config['batch_size'])
    generator, discriminator = create_models(
        latent_dim=config['latent_dim'],
        base_channels=config['base_channels'],
    )
    trainer = NotebookTrainer(
        generator,
        discriminator,
        dataset,
        batch_size=config['batch_size'],
        latent_dim=config['latent_dim'],
        n_critic=config['n_critic'],
        gp_weight=config['gp_weight'],
        lr=config['lr'],
        beta1=config['beta1'],
        beta2=config['beta2'],
        output_dir=config['output_dir'],
        sample_every=config['sample_every'],
        sample_grid=config['sample_grid'],
        max_batches=config.get('max_batches', 0),
        eager=config.get('eager', False),
    )
    history = trainer.train(config['epochs'])
    history_path = Path(config['output_dir']) / 'history.png'
    plot_history(history, title=title, out_path=history_path)
    if history_path.exists():
        print(f'Saved training curves to {history_path}')
    samples_path = Path(config['output_dir']) / 'final_samples.png'
    if samples_path.exists():
        display(Image(filename=str(samples_path)))
        print(f'Saved sample grid to {samples_path}')
    else:
        print(f'Expected samples at {samples_path} but file was not found.')
    return history


In [ ]:
# === Quick Preview ===
train_dataset = get_mnist_dataset(batch_size=64)
preview_batch(train_dataset, n=16)


In [ ]:
baseline_config = {
    'latent_dim': 128,
    'batch_size': 64,
    'n_critic': 3,
    'gp_weight': 10.0,
    'lr': 0.0002,
    'beta1': 0.0,
    'beta2': 0.99,
    'epochs': 20,
    'base_channels': 96,
    'seed': 1234,
    'output_dir': 'outputs/wgan_gp_baseline',
    'sample_every': 5,
    'sample_grid': 6,
    'max_batches': 5,
    'log_interval': 5,
    'eager': True,
}
tf.random.set_seed(baseline_config['seed'])
run_notebook_experiment(baseline_config, 'Baseline WGAN-GP (n_critic=3, λ=10)')


In [ ]:
ncritic1_config = {
    'latent_dim': 128,
    'batch_size': 64,
    'n_critic': 1,
    'gp_weight': 10.0,
    'lr': 0.0002,
    'beta1': 0.0,
    'beta2': 0.99,
    'epochs': 10,
    'base_channels': 96,
    'seed': 1234,
    'output_dir': 'outputs/wgan_gp_ncritic1',
    'sample_every': 5,
    'sample_grid': 6,
    'max_batches': 5,
    'log_interval': 5,
    'eager': True,
}
tf.random.set_seed(ncritic1_config['seed'])
run_notebook_experiment(ncritic1_config, 'WGAN-GP with n_critic=1')


In [ ]:
no_gp_config = {
    'latent_dim': 128,
    'batch_size': 64,
    'n_critic': 3,
    'gp_weight': 0.0,
    'lr': 0.0002,
    'beta1': 0.0,
    'beta2': 0.99,
    'epochs': 15,
    'base_channels': 96,
    'seed': 1234,
    'output_dir': 'outputs/wgan_gp_no_gp',
    'sample_every': 5,
    'sample_grid': 6,
    'max_batches': 5,
    'log_interval': 5,
    'eager': True,
}
tf.random.set_seed(no_gp_config['seed'])
run_notebook_experiment(no_gp_config, 'WGAN without Gradient Penalty')
